<a href="https://colab.research.google.com/github/acac2495/3D_BCH_Parallel_Decoder/blob/main/3D_bch_parallel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Libraries

import numpy as np
import argparse
import itertools
import random

In [ ]:
#Basic building block functions

ALPHA_POWERS = [8, 4, 2, 1, 12, 6, 3, 13, 10, 5, 14, 7, 15, 11, 9]  # a^0..a^14, MSB-first

EXP_STR = {0: "0"}
for idx, val in enumerate(ALPHA_POWERS):
    EXP_STR[val] = f"a^{idx}"

def alpha_pow(exp, n=15):
    return ALPHA_POWERS[exp % n]

def to_hex(val):
    return format(val, 'X')

#XOR array
def xor_array(c1, c2):
    c = [[0] * len(c1) for _ in range(len(c1))]
    for i in range(0, len(c1)):
        for j in range(0, len(c1[0])):
            c[i][j] = c1[i][j] ^ c2[i][j]
    return c

In [ ]:
#Used Constants

N = 15
t = 2
m = 4

In [ ]:
#Galois field operations

def gf_mul(a, b):
    if(a == 0 or b == 0):
        return 0
    p1 = ALPHA_POWERS.index(a)
    p2 = ALPHA_POWERS.index(b)
    return ALPHA_POWERS[(p1 + p2) % N]

def gf_pow(a, pow):
    if(a == 0):
        return 8
    p1 = ALPHA_POWERS.index(a)
    return ALPHA_POWERS[(p1 * pow) % N]

def gf_inv(a):
    if(a == 0):
        return 0
    p1 = ALPHA_POWERS.index(a)
    return ALPHA_POWERS[(N - p1) % N]

def alpha_pow(exp):
    return ALPHA_POWERS[exp % N]

In [ ]:
#Discrete Finite Field Fourier Transform

def dffft_pt(c, j, jp):
    acc = 0
    for i in range(N):
        for ip in range(N):
            if(c[i][ip]):
                acc ^= alpha_pow(i * j + ip * jp)
    return acc

def dffft_2t(c):
    C = [[0] * (2*t) for _ in range(2*t)]
    for j in range(1, 2*t + 1):
        for jp in range(1, 2*t + 1):
            C[j-1][jp-1] = dffft_pt(c, j, jp)
    return C

def dffft_full(c):
    C = [[0] * N for _ in range(N)]
    for j in range(1, N + 1):
        for jp in range(1, N + 1):
            C[j-1][jp-1] = dffft_pt(c, j, jp)
    return C

In [ ]:
#Inverse Discrete Finite Field Fourier Transform

def idffft_pt(C, i, ip):
    acc = 0
    for j in range(N):
        for jp in range(N):
            p1 = gf_inv(alpha_pow(i * j + ip * jp))
            acc ^= gf_mul(p1, C[j][jp])
    if(acc != 0):
        return 1
    else:
        return 0

def idffft_full(C):
    c = [[0] * N for _ in range(N)]
    for i in range(N):
        for ip in range(N):
            c[i][ip] = idffft_pt(C, i, ip)
    return c

In [ ]:
#Printing aids

def print_grid(C):
    for row in C:
        for elem in row:
            print(EXP_STR[elem], " | " , end=" ")
        print("")

def print_bin(err):
    for row in err:
        print(row)

In [ ]:
#Fixed Berlekamp Massey Engine

def BM_fixed(synds):
    sigma   = [0] * (t + 1);  sigma[0] = 8
    sigma_p = [0] * (t + 1);  sigma_p[1] = 8
    d_p = 8
    L = 0
    temp_reg = [0] * (t + 1)
    for i in range(2 * t):
        temp_reg = [synds[i]] + temp_reg[0:t]
        d_mu = temp_reg[0]
        for j in range(1, t + 1):
            d_mu ^= gf_mul(temp_reg[j], sigma[j])

        dmu_dpinv = gf_mul(d_mu, gf_inv(d_p))
        corr = [gf_mul(sigma_p[j], dmu_dpinv) for j in range(t + 1)]
        sigma_next = [sigma[j] ^ corr[j] for j in range(t + 1)]

        load_sigma = 1 if (d_mu != 0 and 2 * L <= i) else 0
        sigma_store = sigma[:] if load_sigma else sigma_p[:]
        sigma_p = [0] + sigma_store[0:t]
        sigma = sigma_next
        if load_sigma:
            d_p = d_mu
            L = i + 1 - L
    return sigma, L

In [ ]:
#Polynomial and Root functions

def print_poly(sigma):
    powers = []
    for coeff in sigma:
        powers.append(EXP_STR[coeff])
    print(powers)

def print_roots(sigma):
    for i in ALPHA_POWERS:
        res = 8
        for j in range(1, t+1):
            pow = gf_pow(i, j)
            res ^= gf_mul(pow, sigma[j])
        if(res == 0):
            print(EXP_STR[i], end=" ")

def display_roots(roots):
    for root in roots:
        print(EXP_STR[root], end=" ")
    print("")

def get_rpos(sigma):
    rpos = []
    for i in ALPHA_POWERS:
        res = 8
        for j in range(1, t+1):
            pow = gf_pow(i, j)
            res ^= gf_mul(pow, sigma[j])
        if(res == 0):
            rpos.append(1)
        else:
            rpos.append(0)
    return rpos

def root_lookup(rpos):
    count = 0
    roots = []
    for i in range(len(rpos)):
        if(rpos[i] == 1):
            roots.append(ALPHA_POWERS[i])
            count += 1
        if(count == t):
            break
    return roots

def root_to_coeff(roots):
    if len(roots) >= 2:
        a1, a2 = roots[0], roots[1]
        prod = gf_mul(a1, a2)
        return [8, gf_mul(a1 ^ a2, gf_inv(prod)), gf_inv(prod)]
    if len(roots) == 1:
        return [8, gf_inv(roots[0]), 0]
    return [8, 0, 0]

In [ ]:
#Recursive Extension basic functions

def RE_base(c, window):
    res = 0
    for i in range(1, t + 1):
        res ^= gf_mul(c[i], window[-i])
    return res

def RE_chain(c, S_init):
    window = list(S_init)
    out = []
    for _ in range(N - 2 * t):
        new = RE_base(c, window)
        out.append(new)
        window = window[1:] + [new]
    return out


In [ ]:
#Recursive extension row and column wise

def extend_columns(S_2t, col_ccp_poly):
    S_full = [[0] * (2 * t) for _ in range(N)]
    for k in range(2*t):
        seq = [S_2t[j][k] for j in range(2*t)]
        seq_full = seq + RE_chain(col_ccp_poly, seq[-t:])
        for j in range(N):
            S_full[j][k] = seq_full[j]
    return S_full

def extend_rows(col_full, row_ccp_poly):
    S_full = []
    for j in range(N):
        seq = list(col_full[j])
        S_full.append(seq + RE_chain(row_ccp_poly, seq[-t:]))
    return S_full

In [ ]:
#Full datapath implementation

def ecc_2d_silent(r_in):
    S_2t = dffft_2t(r_in)
    S_2t_arrnp = np.array(S_2t)
    S_2t_colwise = np.transpose(S_2t_arrnp).tolist()

    col_poly = []
    row_poly = []
    col_deg = []
    row_deg = []

    for j in S_2t:
        sigma_j, deg_j = BM_fixed(j)
        row_poly.append(sigma_j)
        row_deg.append(deg_j)
    for j in S_2t_colwise:
        sigma_j, deg_j = BM_fixed(j)
        col_poly.append(sigma_j)
        col_deg.append(deg_j)

    col_rpos = []
    row_rpos = []

    for sigma in col_poly:
        col_rpos.append(get_rpos(sigma))
    for sigma in row_poly:
        row_rpos.append(get_rpos(sigma))

    col_rpos_lcm = 0
    row_rpos_lcm = 0

    col_rpos_lcm = [0] * N
    row_rpos_lcm = [0] * N

    for rpos in col_rpos:
        col_rpos_lcm = [a | b for a, b in zip(col_rpos_lcm, rpos)]
    for rpos in row_rpos:
        row_rpos_lcm = [a | b for a, b in zip(row_rpos_lcm, rpos)]

    roots_row = root_lookup(row_rpos_lcm)
    roots_col = root_lookup(col_rpos_lcm)

    col_ccp_poly = root_to_coeff(roots_col)
    row_ccp_poly = root_to_coeff(roots_row)

    col_full = extend_columns(S_2t, col_ccp_poly)
    row_full = extend_rows(col_full, row_ccp_poly)

    err_obtained = idffft_full(row_full)

    return err_obtained


In [ ]:
#Used Constants

XY = 0
XZ = 1
YZ = 2
MAX = 100

In [ ]:
import numpy as np

def cube_to_hex(cube):
    """Hex of the cube as a [N*N*N-1:0] value: bit (x*N+y)*N+z = cube[x,y,z], MSB first."""
    cube = np.asarray(cube).astype(np.uint8)
    nbits = cube.size
    bits = cube.reshape(-1)                         # C order: index = (x*N + y)*N + z
    value = int("".join(map(str, bits[::-1])), 2)   # bit 0 on the right
    return f"{value:0{(nbits + 3) // 4}x}"          # 3375 bits -> 844 digits, zero padded

def hex_to_cube(h, N=15):
    value = int(h.strip(), 16)
    bits = [(value >> k) & 1 for k in range(N**3)]
    return np.array(bits, dtype=np.uint8).reshape(N, N, N)

def print_cube_hex(cube, label="cube", width=64):
    h = cube_to_hex(cube)
    print(f"{label} [{np.asarray(cube).size-1}:0] = {len(h)*4}'h")
    for k in range(0, len(h), width):
        print("  " + h[k:k+width])

def write_cube_hex(cube, path):
    """One line, no prefix: readable by $readmemh into a [N*N*N-1:0] word."""
    with open(path, "w") as f:
        f.write(cube_to_hex(cube) + "\n")

In [ ]:
#3d datapath implementation

def display_orientation(o):
    if(o == XY):
        return "XY"
    elif(o == XZ):
        return "XZ"
    elif(o == YZ):
        return "YZ"
    raise ValueError(f"bad orientation {o}")

def display_slice_err(slc, err, o):
    print(f"Orientation : {display_orientation(o)}")
    print(f"slice : ")
    print_bin(slc)
    print(f"error : ")
    print_bin(err)

def extract_slice(cube, o, i):
    if(o == XY):
        return cube[:, :, i]
    elif(o == XZ):
        return cube[:, i, :]
    elif(o == YZ):
        return cube[i, :, :]

def check_block(slc, err):
    if ((slc ^ err) == 0).all():
        return True
    else:
        return False

def check_block_syndrome(slc, err):
    synd = np.array(dffft_2t(slc ^ err))
    return not synd.any()

def single_pass(cube, o):
    dirty = False
    slices = []
    for i in range (N):
        slices.append(extract_slice(cube, o, i))
    errors = []
    for i in slices:
        errors.append(ecc_2d_silent(i))
    corrs = []
    for i in range(N):
        if(check_block_syndrome(slices[i], errors[i]) and slices[i].any()):
            #display_slice_err(slices[i], errors[i], o)
            corrs.append(slices[i] ^ errors[i])
            if(dirty == False):
                dirty = True
        else:
            corrs.append(slices[i])

    if o == XY:
        corrected_cube = np.stack(corrs, axis=2)
    elif o == XZ:
        corrected_cube = np.stack(corrs, axis=1)
    elif o == YZ:
        corrected_cube = np.stack(corrs, axis=0)

    return corrected_cube, dirty

def full_datapath_3d(cube):
    iters = 0
    while(iters < MAX):
        sweep_dirty = False
        for o in range(3):
            cube, dirty = single_pass(cube, o)
            print_cube_hex(cube, f"iter {iters} pass {display_orientation(o)}")
            #print(f"Done iter : {iters}, pass : {display_orientation(o)}, Dirty : {dirty}")
            if(dirty):
                if(sweep_dirty == False):
                    sweep_dirty = True
        #print(f"Done iter : {iters}, Dirty Sweep : {sweep_dirty}")
        if(sweep_dirty == False):
            break
        iters += 1
    return cube, iters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_cube(cube, title="cube", prev=None, ax=None, show=True):
    """3D view of an NxNxN 0/1 cube, indexed cube[x, y, z].
    Without prev: every set bit is a red dot.
    With prev (the cube before this pass):
        red dot    = still set (error not yet fixed)
        green ring = bit cleared by this pass
        black X    = bit newly set by this pass (a miscorrection)
    """
    cube = np.asarray(cube).astype(bool)
    own = ax is None
    if own:
        fig = plt.figure(figsize=(6.5, 6))
        ax = fig.add_subplot(111, projection="3d")
    N = cube.shape[0]
    if prev is None:
        x, y, z = np.nonzero(cube)
        ax.scatter(x, y, z, c="crimson", s=28, depthshade=True, label=f"set bits ({len(x)})")
    else:
        prev = np.asarray(prev).astype(bool)
        keep, cleared, new = cube & prev, prev & ~cube, cube & ~prev
        for m, kw in ((keep,    dict(c="crimson", marker="o", s=28, label=f"still set ({keep.sum()})")),
                      (cleared, dict(facecolors="none", edgecolors="limegreen", marker="o", s=70, linewidths=1.8, label=f"cleared ({cleared.sum()})")),
                      (new,     dict(c="black", marker="x", s=45, label=f"newly set ({new.sum()})"))):
            x, y, z = np.nonzero(m)
            if len(x):
                ax.scatter(x, y, z, **kw)
    ax.set_xlim(-0.5, N - 0.5); ax.set_ylim(-0.5, N - 0.5); ax.set_zlim(-0.5, N - 0.5)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_xticks(range(0, N, 2)); ax.set_yticks(range(0, N, 2)); ax.set_zticks(range(0, N, 2))
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    if own and show:
        plt.tight_layout(); plt.show()
    return ax

def show_slices(cube, o, title=None, only_nonzero=False, show=True):
    """Grid of the N slices for one orientation (0=XY, 1=XZ, 2=YZ).
    Same slicing as extract_slice(): XY=cube[:,:,i], XZ=cube[:,i,:], YZ=cube[i,:,:]."""
    cube = np.asarray(cube)
    N = cube.shape[0]
    name = ("XY", "XZ", "YZ")[o]
    idx = list(range(N))
    sl = lambda i: (cube[:, :, i], cube[:, i, :], cube[i, :, :])[o]
    if only_nonzero:
        idx = [i for i in idx if sl(i).any()] or [0]
    cols = min(5, len(idx)); rows = -(-len(idx) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(2.3 * cols, 2.5 * rows), squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for k, i in enumerate(idx):
        a = axes[k // cols][k % cols]; a.axis("on")
        s = sl(i)
        a.imshow(s, cmap="Reds", vmin=0, vmax=1, interpolation="nearest")
        a.set_title(f"{name}[{i}]  ({int(s.sum())})", fontsize=9)
        a.set_xticks([]); a.set_yticks([])
        for sp in a.spines.values():
            sp.set_edgecolor("crimson" if s.any() else "lightgray")
    fig.suptitle(title or f"{name} slices (row = first axis, col = second axis)", fontsize=11)
    plt.tight_layout()
    if show: plt.show()
    return fig

def show_pass(prev, cur, title, o=None, slices=True):
    """One call per pass: 3D diff view, plus the slices of that pass's orientation if wanted."""
    show_cube(cur, title, prev=prev)
    if slices and o is not None:
        show_slices(cur, o, title=f"{title} -- {('XY','XZ','YZ')[o]} slices after this pass", only_nonzero=True)


def full_datapath_3d_viz(cube, MAX=100, slices=True):
    """Same loop as full_datapath_3d, but shows the cube after every pass.
    Pass in your (fixed) single_pass function."""
    names = ("XY", "XZ", "YZ")
    show_cube(cube, "input cube")
    iters = 0
    while iters < MAX:
        sweep_dirty = False
        for o in range(3):
            prev = cube.copy()
            cube, dirty = single_pass(cube, o)
            sweep_dirty |= dirty
            print(f"iter {iters}, pass {names[o]}: dirty={dirty}, bits set {int(prev.sum())} -> {int(cube.sum())}")
            show_pass(prev, cube, f"iter {iters}, after {names[o]} pass", o=o, slices=slices)
        if not sweep_dirty:
            break
        iters += 1
    return cube, iters

In [ ]:
#Test with a cube :

test_cube = np.zeros((N, N, N), dtype=np.uint8)

# Introduce sparse errors (up to t=2 errors per plane)

test_cube[1, 8, 5] = 1
test_cube[1, 12, 5] = 1
test_cube[2, 8, 5] = 1
test_cube[2, 10, 5] = 1
test_cube[2, 12, 5] = 1
test_cube[4, 8, 5] = 1
test_cube[4, 12, 5] = 1
test_cube[5, 8, 5] = 1
test_cube[5, 12, 5] = 1

test_cube[1, 8, 10] = 1
test_cube[1, 12, 10] = 1
test_cube[2, 8, 10] = 1
test_cube[2, 10, 10] = 1
test_cube[2, 12, 10] = 1
test_cube[4, 8, 10] = 1
test_cube[4, 12, 10] = 1
test_cube[5, 8, 10] = 1
test_cube[5, 12, 10] = 1

test_cube[8, 8, 5] = 1
test_cube[8, 12, 5] = 1
test_cube[9, 8, 5] = 1
test_cube[9, 10, 5] = 1
test_cube[9, 12, 5] = 1
test_cube[11, 8, 5] = 1
test_cube[11, 12, 5] = 1
test_cube[12, 8, 5] = 1
test_cube[12, 12, 5] = 1

test_cube[8, 8, 10] = 1
test_cube[8, 12, 10] = 1
test_cube[9, 8, 10] = 1
test_cube[9, 10, 10] = 1
test_cube[9, 12, 10] = 1
test_cube[11, 8, 10] = 1
test_cube[11, 12, 10] = 1
test_cube[12, 8, 10] = 1
test_cube[12, 12, 10] = 1

print_cube_hex(test_cube, "initial cube")
corrected_cube, iters = full_datapath_3d(test_cube)

initial cube [3374:0] = 3376'h
  0000000000000000000000000000000000000000000000000000000000000000
  0000000000000000000000000000000000000000000000000000000004200000
  0000000042000000000000000000000000000000000000000210000000000000
  2100000000000000000000000000000000000000000000000000000000000000
  0000000000000000000000000000000000840000021000000840000000000000
  0000000000000000000000000042000000000000042000000000000000000000
  0000000000000000000000000000000000000000000000000000000000000000
  0000000000000000000000000000000000000000000000000000000000000000
  0008400000000000008400000000000000000000000000000000000000042000
  0000000000420000000000000000000000000000000000000000000000000000
  0000000000000000000000000000000000000000000108000004200000108000
  0000000000000000000000000000000000008400000000000008400000000000
  0000000000000000000000000000000000000000000000000000000000000000
  000000000000
iter 0 pass XY [3374:0] = 3376'h
  000000000000000000000000000000000000000000000000

In [ ]:
test_cube = np.zeros((N, N, N), dtype=np.uint8)

# Burst A: 2 x 2 x 4 block. Every XY slice sees a 2x2 square -> fixed in the XY pass
test_cube[13:15, 0:2, 0:4] = 1

# Burst B: 2 x 4 x 2 block. XY slices see 2x4 (too wide), XZ slices see a 2x2 square -> fixed in the XZ pass
test_cube[6:8, 2:6, 7:9] = 1

# Four copies of the pattern from before: XY slices are too spread out to fix,
# the XZ pass fixes only the y=10 slices, which lets the YZ pass fix the rest
test_cube[1, 8, 5] = 1
test_cube[1, 12, 5] = 1
test_cube[2, 8, 5] = 1
test_cube[2, 10, 5] = 1
test_cube[2, 12, 5] = 1
test_cube[4, 8, 5] = 1
test_cube[4, 12, 5] = 1
test_cube[5, 8, 5] = 1
test_cube[5, 12, 5] = 1

test_cube[1, 8, 10] = 1
test_cube[1, 12, 10] = 1
test_cube[2, 8, 10] = 1
test_cube[2, 10, 10] = 1
test_cube[2, 12, 10] = 1
test_cube[4, 8, 10] = 1
test_cube[4, 12, 10] = 1
test_cube[5, 8, 10] = 1
test_cube[5, 12, 10] = 1

test_cube[8, 8, 5] = 1
test_cube[8, 12, 5] = 1
test_cube[9, 8, 5] = 1
test_cube[9, 10, 5] = 1
test_cube[9, 12, 5] = 1
test_cube[11, 8, 5] = 1
test_cube[11, 12, 5] = 1
test_cube[12, 8, 5] = 1
test_cube[12, 12, 5] = 1

test_cube[8, 8, 10] = 1
test_cube[8, 12, 10] = 1
test_cube[9, 8, 10] = 1
test_cube[9, 10, 10] = 1
test_cube[9, 12, 10] = 1
test_cube[11, 8, 10] = 1
test_cube[11, 12, 10] = 1
test_cube[12, 8, 10] = 1
test_cube[12, 12, 10] = 1


print_cube_hex(test_cube, "initial cube")
corrected_cube, iters = full_datapath_3d(test_cube)

initial cube [3374:0] = 3376'h
  0000000000000000000000000000000000000000000000000001e003c0000000
  00000000000000000000000000000000000000000000f001e000000004200000
  0000000042000000000000000000000000000000000000000210000000000000
  2100000000000000000000000000000000000000000000000000000000000000
  0000000000000000000000000000000000840000021000000840000000000000
  0000000000000000000000000042000000000000042000000000000000000000
  0000000000000000000000000000000000000000000006000c00180030000000
  000000000000000000000000000000000000030006000c001800000000000000
  0008400000000000008400000000000000000000000000000000000000042000
  0000000000420000000000000000000000000000000000000000000000000000
  0000000000000000000000000000000000000000000108000004200000108000
  0000000000000000000000000000000000008400000000000008400000000000
  0000000000000000000000000000000000000000000000000000000000000000
  000000000000
iter 0 pass XY [3374:0] = 3376'h
  000000000000000000000000000000000000000000000000